# Nhúng phụ đề bằng BGE-M3 — Batch 2

`config/ingest_shards_batch2_btc_subtitles.yaml`


## Step 0a — repo + gói

In [ ]:
MEMBER      = "trong"          # <-- ĐỔI THÀNH TÊN BẠN

SHARDS      = "config/ingest_shards_batch2_btc_subtitles.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-batch-2-v2"  # CÙNG volume với notebook 02/03/04/06/08
VOL_VERSION = 2

# ── CHẠY THỬ / CHẠY LẠI ───────────────────────────────────────────────────
ONLY_VIDEOS = []   # [] = cả phần của bạn. Đặt vài tên để thử trước.
REDO_STAGES = []   # chạy lại DÙ marker đã báo xong (ghi đè part + marker cũ)

REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
# ⚠️ MỌI NGƯỜI CHẠY BUNDLE NÀY PHẢI ĐỂ CÙNG MỘT NHÁNH — xem notebook 03.
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)   # token không lọt vào log
sys.path.insert(0, str(REPO / "src"))

req = REPO / "notebooks" / "requirements-kaggle-btc-bge.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

# `sentence-transformers` va `torch` deu nam trong NUMERIC_PACKAGES va trong
# STAGE_PACKAGES["n6_bge_m3"]: chung quyet dinh chinh con so trong vector.
# `merge_bundle.py` TU CHOI bundle co hai ban khac nhau — va no chi noi ra o
# buoc gop. Chep hai con so duoi day cho nguoi chay phan con lai.
from importlib.metadata import version

print("\n>>> CHÉP HAI CON SỐ NÀY cho người chạy phần còn lại <<<")
for package in ("sentence-transformers", "torch"):
    try:
        print(f"      {package}=={version(package)}")
    except Exception as err:      # noqa: BLE001 - chi de bao, khong de chan
        print(f"      {package}: KHÔNG ĐỌC ĐƯỢC ({err})")

# vLLM khong duoc phep co mat: no keo torch sang ban khac. Kiem ngay bay gio
# ton 0 giay; phat hien o buoc gop ton nhieu gio cua ca nhom.
try:
    bad = version("vllm")
except Exception:
    bad = None
if bad is not None:
    raise SystemExit(
        f"vllm=={bad} có trong phiên này. Notebook 09 KHÔNG được chạy chung phiên với 08:\n"
        "  vLLM ghim torch rất chặt, và torch quyết định vector BGE-M3 sinh ra.\n"
        "  Sửa: tạo một phiên Kaggle MỚI và chỉ chạy notebook 09 trong đó."
    )

## Step 0b — Modal, phân công, điều kiện tiên quyết

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import yaml
from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, unload_models
from mfe.offline.transport import build_transport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
STAGE_LIST = list(plan.stages)          # ['n6_bge_m3']
PARAMS     = RAW.get("params", {})

MY_VIDEOS = list(plan.videos_for(MEMBER))
status    = StageStatus.load(transport, layout, MEMBER)

# ĐIỀU KIỆN TIÊN QUYẾT — và đây là phép đếm đáng tin duy nhất.
#
# `StageStatus` đọc marker THEO TỪNG NGƯỜI (CLAUDE.md §5), nên con số dưới đây là
# "caption của PHẦN VIỆC CỦA BẠN đã xong tới đâu", không phải của cả bundle. Đó
# đúng là con số cần: bạn chỉ nhúng được phần của chính mình.
ready   = [v for v in MY_VIDEOS if status.is_done("n2_subtitles", v)]
waiting = [v for v in MY_VIDEOS if v not in set(ready)]

wanted = set(ONLY_VIDEOS) & set(MY_VIDEOS) if ONLY_VIDEOS else set(MY_VIDEOS)
todo = [v for v in ready
        if v in wanted and (REDO_STAGES or not status.is_done("n6_bge_m3", v))]

print()
print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"video   : {len(MY_VIDEOS)} được giao")
print(f"caption : {len(ready)} đã có (lượt 5 xong), {len(waiting)} còn chờ")
print(f"còn lại : {len(todo)} video cần nhúng")
if waiting:
    print(f"\nCHU Y: {len(waiting)} video CHUA co caption — se BO QUA, khong phai loi.")
    print(f"       vi du: {', '.join(waiting[:6])}")
if ONLY_VIDEOS:
    print(f"\n>>> ONLY_VIDEOS: chi chay {len(todo)} video <<<")

## Step 1 — nhúng

Một vòng lặp phẳng theo video. Không tải zip, không bung ảnh, không dọn đĩa — nên cũng không có
khối `try/finally` dọn dẹp nào như notebook 08.

`StageContext` vẫn cần một danh sách keyframe vì đó là hình dạng chung của mọi tầng, nhưng
`stage_n6_bge_m3` **không đọc nó**: khoá của vector đến từ khoá của caption. Truyền `[]` vào là
cách nói ra điều đó — một danh sách keyframe giả ở đây sẽ trông như một phép tra cứu đang xảy ra.

Model nạp **một lần** cho cả phiên qua `cached_model`, và bạn đọc được điều đó trong log: dòng
`nap model bge-m3 BAAI/bge-m3` phải xuất hiện **đúng một lần**, ở video đầu tiên. Thấy nó lặp
lại ở video thứ hai là cache đã hỏng — dừng lại, đừng để chạy tiếp.

Vì sao đáng canh: BGE-M3 nặng ~2,2 GB còn phần việc thật của một video chỉ là nhúng vài trăm câu
ngắn. Nạp lại theo từng video là 404 lần nạp với `trong`, 469 với `quang` — **5–16 giờ chỉ để
nạp model**, đo được ở các tầng khác (`tests/.../test_multipass_bundle.py::TestModelCache`). Và
nó không ồn ào chút nào: mỗi video vẫn xong, marker vẫn cắm, chỉ lâu gấp bội.

Đây từng là lỗi thật của chính tầng này — `stage_n6_bge_m3` là tầng **duy nhất** gọi
`SentenceTransformer(...)` thẳng trong thân hàm thay vì qua `cached_model`. Sửa 26-08-2026.

`unload_models()` ở Step 2 mới thả model ra.

In [ ]:
import json, time, traceback

def keyframe_count(video_id, fallback):
    """Số keyframe THẬT của video, đọc từ manifest `n0_import_btc` đã ghi.

    KHÔNG truyền số vector vào chỗ này. `write_part_manifest` ghi
    `counts.keyframes` từ tham số truyền vào và **không gộp** trường đó như nó
    gộp `artifacts` — nên một con số khác sẽ **ghi đè** con số `n0_import_btc`
    đã đo, theo kiểu không ai thấy: manifest vẫn hợp lệ, chỉ nói sai.

    Ở đây hai con số *thường* bằng nhau (một caption mỗi keyframe), và chính vì
    thường bằng nhau nên sai lệch sẽ không lộ ra. Đọc chứ đừng suy ra.
    Cùng khuôn với `scripts/rebuild_artifact_manifest.py`.
    """
    local = WORK / f"cnt_{video_id}.json"
    try:
        transport.get(layout.part_manifest(video_id), local)
        return int(json.loads(local.read_text(encoding="utf-8"))["counts"]["keyframes"])
    except Exception:
        return fallback
    finally:
        local.unlink(missing_ok=True)


failures = []
started = time.perf_counter()
rows_total = 0

if not todo:
    print("khong co viec — Step 0b da noi ly do.")
else:
    for index, video_id in enumerate(todo, 1):
        try:
            # keyframes=[] CO CHU Y: tang nay khong doc chung. Xem o markdown tren.
            ctx = StageContext(video_id, Path("/dev/null"), [], writer, layout, WORK)
            ctx.params = dict(PARAMS.get("n7_bge_m3_subtitles", {}))
            marker = STAGES["n7_bge_m3_subtitles"](ctx)

            rows = marker.get("rows", 0)
            writer.write_part_manifest(
                video_id, [marker],
                keyframes=keyframe_count(video_id, rows), merge=True,
            )
            status.done.add(("n7_bge_m3_subtitles", video_id))
            rows_total += rows

            elapsed = time.perf_counter() - started
            left = (len(todo) - index) * elapsed / index
            print(f"  [{index:>3}/{len(todo)}] {video_id}  {rows:>5} vector  "
                  f"{marker.get('elapsed_s')}s  | còn ~{left / 60:.0f} phút")
        except Exception as err:
            failures.append((video_id, repr(err)))
            print(f"  [LỖI] {video_id}: {err}")
            traceback.print_exc(limit=3)
        finally:
            (WORK / f"subtitles_segments_{video_id}.json").unlink(missing_ok=True)
            (WORK / f"dense_vectors_{video_id}.npz").unlink(missing_ok=True)

    print()
    print("=" * 72)
    print(f"phiên này: {len(todo) - len(failures)} video, {rows_total:,} vector, "
          f"{len(failures)} lỗi, {(time.perf_counter() - started) / 60:.0f} phút")
    for video_id, err in failures:
        print(f"  {video_id}: {err}")

## Step 2 — chốt phiên

`StageStatus.load` lại từ Volume chứ không đọc biến trong bộ nhớ. Ba lần trong dự án này câu trả
lời đúng chỉ đến khi **đếm dữ liệu**, không phải khi đọc log (CLAUDE.md §5).

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
done  = [v for v in MY_VIDEOS if fresh.is_done("n7_bge_m3_subtitles", v)]
cap   = [v for v in MY_VIDEOS if fresh.is_done("n2_subtitles", v)]
left  = [v for v in cap if v not in set(done)]

print(f"{MEMBER}: {len(done)}/{len(MY_VIDEOS)} video đã nhúng và lên {VOLUME}")
print(f"         {len(cap)}/{len(MY_VIDEOS)} video đã có caption (trần của lượt này)")

if left:
    print(f"\ncòn {len(left)} video có caption mà chưa nhúng — bấm Run All lại, nó chạy tiếp")
elif len(cap) < len(MY_VIDEOS):
    print(f"\nHẾT phần nhúng được. {len(MY_VIDEOS) - len(cap)} video còn chờ notebook 08.")
else:
    print("\nXONG cả phần của bạn. Báo Lead.")
    print(f"  python scripts/verify_bundle.py --shards {SHARDS} --member {MEMBER}")
    print(f"  python scripts/merge_bundle.py  --shards {SHARDS}")